# Multi-Metric Agreement Evaluation

1. **Cohen's κ** — the standard metric, kept for continuity with prior work.
2. **Gwet's AC1** — chance-corrected agreement that is robust to label imbalance (Gwet, 2008).
3. **Problem-level F1** — agreement at the instructor-relevant granularity (did raters identify the same gap set for this student on this problem?).

In [8]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score, f1_score, precision_score, recall_score
from irrCAC.raw import CAC

# 18 KC tags — must match the annotation tool and prompt vocabulary exactly.
EXACT_KC_TAGS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]

# Student IDs covered in the annotation study (one per cluster)
STUDENT_IDS = ['10155', '14475', '14476']  # Struggling, High Performer, Average

# Each rater has three JSON files — one per student. They get merged at load time.
ANNOTATION_PATHS = {
    'Human_A': [
        "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_10155_1774736175604.json",
        "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14475_1775593592812.json",
        "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14476_1775593175294.json",
    ],
    'Human_B': [
        "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_10155_1774820622134.json",
        "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14475_1775593132134.json",
        "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14476_1775691915454.json",
    ],
    'Exp10a_Baseline': [
        "results/human_validation/llm_baseline_annotations_10155.json",
        "results/human_validation/llm_baseline_annotations_14475.json",
        "results/human_validation/llm_baseline_annotations_14476.json",
    ],
    'Exp10b_Enriched': [
        "results/human_validation/llm_annotations_10155.json",
        "results/human_validation/llm_annotations_14475.json",
        "results/human_validation/llm_annotations_14476.json",
    ],
    'Exp11_Baseline_V2': [
        "results/human_validation/llm_baseline_annotations_v2_10155.json",
        "results/human_validation/llm_baseline_annotations_v2_14475.json",
        "results/human_validation/llm_baseline_annotations_v2_14476.json",
    ],
    'Exp11_Enriched_V2': [
        "results/human_validation/llm_enriched_annotations_v2_10155.json",
        "results/human_validation/llm_enriched_annotations_v2_14475.json",
        "results/human_validation/llm_enriched_annotations_v2_14476.json",
    ],
    'V3': [
        "results/human_validation/llm_v3_annotations_10155.json",
        "results/human_validation/llm_v3_annotations_14475.json",
        "results/human_validation/llm_v3_annotations_14476.json",
    ],
}

In [9]:
def load_annotations(paths: list[str]) -> dict[str, set[str]]:
    """Load and merge annotation JSON files from multiple students.
    
    Each file covers one student. We merge by namespacing problem_ids 
    with the student_id to keep them disjoint across students.
    Returns {student_id:problem_id: set(gaps)}.
    """
    merged = {}
    for path in paths:
        # Extract student_id from filename (e.g., "..._10155_..." -> "10155")
        # The student_id is the numeric segment matching one of STUDENT_IDS.
        student_id = next((sid for sid in STUDENT_IDS if f'_{sid}_' in path or f'_{sid}.' in path), None)
        if student_id is None:
            raise ValueError(f"Could not extract student_id from {path}")
        
        with open(path, 'r') as f:
            raw = json.load(f)
        anns = raw.get('annotations', raw)  # support both wrapped and flat
        
        for pid, v in anns.items():
            key = f'{student_id}:{pid}'  # namespace to avoid collision across students
            merged[key] = set(v.get('gaps', []))
    return merged

all_annotations = {name: load_annotations(paths) for name, paths in ANNOTATION_PATHS.items()}

# Common keys across all raters (student:problem pairs annotated by everyone)
common_pids = sorted(set.intersection(*[set(a.keys()) for a in all_annotations.values()]))

print(f'Loaded {len(all_annotations)} annotation sets')
for name, anns in all_annotations.items():
    by_student = {}
    for key in anns:
        sid = key.split(':')[0]
        by_student[sid] = by_student.get(sid, 0) + 1
    print(f'  {name}: {len(anns)} total — ' + ', '.join(f'{sid}:{n}' for sid, n in sorted(by_student.items())))
print(f'\nCommon (student:problem) pairs across all sets: {len(common_pids)}')

Loaded 7 annotation sets
  Human_A: 150 total — 10155:50, 14475:50, 14476:50
  Human_B: 150 total — 10155:50, 14475:50, 14476:50
  Exp10a_Baseline: 146 total — 10155:46, 14475:50, 14476:50
  Exp10b_Enriched: 146 total — 10155:46, 14475:50, 14476:50
  Exp11_Baseline_V2: 146 total — 10155:46, 14475:50, 14476:50
  Exp11_Enriched_V2: 146 total — 10155:46, 14475:50, 14476:50
  V3: 146 total — 10155:46, 14475:50, 14476:50

Common (student:problem) pairs across all sets: 146


In [10]:
def build_rater_matrix(anns_x: dict, anns_y: dict, pids: list) -> pd.DataFrame:
    """Build a DataFrame with one row per (problem, KC) item and two rater columns."""
    rows = []
    for pid in pids:
        gx = anns_x.get(pid, set())
        gy = anns_y.get(pid, set())
        for kc in EXACT_KC_TAGS:
            rows.append({
                'item': f'{pid}_{kc}',
                'rater_x': 1 if kc in gx else 0,
                'rater_y': 1 if kc in gy else 0,
            })
    return pd.DataFrame(rows).set_index('item')

In [11]:
def compute_cell_kappa(df: pd.DataFrame) -> float:
    return cohen_kappa_score(df['rater_x'], df['rater_y'])

def compute_gwet_ac1(df: pd.DataFrame) -> dict:
    """Compute Gwet's AC1 with 95% CI using irrCAC."""
    cac = CAC(df[['rater_x', 'rater_y']])
    result = cac.gwet()
    coeff = result['est']['coefficient_value']
    ci = result['est']['confidence_interval']
    return {'ac1': coeff, 'ci_low': ci[0], 'ci_high': ci[1]}

def compute_problem_level_f1(anns_x: dict, anns_y: dict, pids: list) -> dict:
    """Per-problem set F1, averaged across problems."""
    f1s, jaccards = [], []
    for pid in pids:
        gx, gy = anns_x.get(pid, set()), anns_y.get(pid, set())
        if not gx and not gy:
            f1s.append(1.0); jaccards.append(1.0); continue
        if not gx or not gy:
            f1s.append(0.0); jaccards.append(0.0); continue
        tp = len(gx & gy)
        precision = tp / len(gy) if gy else 0
        recall = tp / len(gx) if gx else 0
        f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0
        jaccard = tp / len(gx | gy)
        f1s.append(f1); jaccards.append(jaccard)
    return {
        'problem_f1_mean': np.mean(f1s),
        'jaccard_mean': np.mean(jaccards),
    }

In [12]:
def evaluate_pair(name_x: str, anns_x: dict, name_y: str, anns_y: dict, pids: list) -> dict:
    df = build_rater_matrix(anns_x, anns_y, pids)
    kappa = compute_cell_kappa(df)
    gwet = compute_gwet_ac1(df)
    plf1 = compute_problem_level_f1(anns_x, anns_y, pids)
    return {
        'Comparison': f'{name_x} vs {name_y}',
        'Cohen_kappa': kappa,
        'Gwet_AC1': gwet['ac1'],
        'AC1_CI_low': gwet['ci_low'],
        'AC1_CI_high': gwet['ci_high'],
        'Problem_F1': plf1['problem_f1_mean'],
        'Jaccard': plf1['jaccard_mean'],
    }

# Human-Human ceiling
results = [evaluate_pair('Human_A', all_annotations['Human_A'],
                          'Human_B', all_annotations['Human_B'], common_pids)]
results[0]['Comparison'] = 'H-A vs H-B (Ceiling)'

# Each LLM config vs each human, and averaged
llm_configs = [
    ('Exp10a_Baseline', 'AvgHuman vs Baseline'),
    ('Exp10b_Enriched', 'AvgHuman vs Enriched'),
    ('Exp11_Baseline_V2', 'AvgHuman vs Baseline_V2'),
    ('Exp11_Enriched_V2', 'AvgHuman vs Enriched_V2'),
    ('V3', 'AvgHuman vs V3'),
]
for cfg, display_name in llm_configs:
    r_a = evaluate_pair('Human_A', all_annotations['Human_A'], cfg, all_annotations[cfg], common_pids)
    r_b = evaluate_pair('Human_B', all_annotations['Human_B'], cfg, all_annotations[cfg], common_pids)
    avg = {
        'Comparison': display_name,
        'Cohen_kappa': (r_a['Cohen_kappa'] + r_b['Cohen_kappa']) / 2,
        'Gwet_AC1': (r_a['Gwet_AC1'] + r_b['Gwet_AC1']) / 2,
        'AC1_CI_low': None,
        'AC1_CI_high': None,  # CIs don't average cleanly; leave blank
        'Problem_F1': (r_a['Problem_F1'] + r_b['Problem_F1']) / 2,
        'Jaccard': (r_a['Jaccard'] + r_b['Jaccard']) / 2,
    }
    results.extend([r_a, r_b, avg])

results_df = pd.DataFrame(results)
results_df

,Comparison,Cohen_kappa,Gwet_AC1,AC1_CI_low,AC1_CI_high,Problem_F1,Jaccard
0,H-A vs H-B (Ceiling),0.574306,0.953110,0.94421,0.96201,0.869015,0.833545
1,Human_A vs Exp10a_Baseline,0.383362,0.907720,0.89483,0.92061,0.756287,0.722554
2,Human_B vs Exp10a_Baseline,0.305417,0.903900,0.89078,0.91701,0.726424,0.693708
3,AvgHuman vs Baseline,0.344390,0.905810,NaN,NaN,0.741356,0.708131
4,Human_A vs Exp10b_Enriched,0.417188,0.935320,0.92480,0.94585,0.792808,0.759257
5,Human_B vs Exp10b_Enriched,0.328041,0.933050,0.92238,0.94372,0.775908,0.748190
6,AvgHuman vs Enriched,0.372614,0.934185,NaN,NaN,0.784358,0.753724
7,Human_A vs Exp11_Baseline_V2,0.470291,0.933630,0.92290,0.94436,0.825186,0.792422
8,Human_B vs Exp11_Baseline_V2,0.382673,0.929640,0.91863,0.94066,0.796908,0.769871
9,AvgHuman vs Baseline_V2,0.426482,0.931635,NaN,NaN,0.811047,0.781147


In [13]:
desired_order = [
    'H-A vs H-B (Ceiling)',
    'AvgHuman vs Baseline',
    'AvgHuman vs Enriched',
    'AvgHuman vs Baseline_V2',
    'AvgHuman vs Enriched_V2',
    'AvgHuman vs V3',
]

thesis_table = results_df[results_df['Comparison'].isin(desired_order)].copy()
thesis_table['Comparison'] = pd.Categorical(thesis_table['Comparison'], categories=desired_order, ordered=True)
thesis_table = thesis_table.sort_values('Comparison')[
    ['Comparison', 'Cohen_kappa', 'Gwet_AC1', 'Problem_F1', 'Jaccard']
].round(3).reset_index(drop=True)

print('\n=== Thesis Table: Agreement Metrics ===\n')
print(thesis_table.to_string(index=False))
thesis_table.to_csv('results/human_llm_agreement/agreement_metrics_table.csv', index=False)
print('\nSaved to: agreement_metrics_table.csv')


=== Thesis Table: Agreement Metrics ===

             Comparison  Cohen_kappa  Gwet_AC1  Problem_F1  Jaccard
   H-A vs H-B (Ceiling)        0.574     0.953       0.869    0.834
   AvgHuman vs Baseline        0.344     0.906       0.741    0.708
   AvgHuman vs Enriched        0.373     0.934       0.784    0.754
AvgHuman vs Baseline_V2        0.426     0.932       0.811    0.781
AvgHuman vs Enriched_V2        0.406     0.927       0.798    0.764
         AvgHuman vs V3        0.509     0.952       0.842    0.811

Saved to: agreement_metrics_table.csv
